# 🎭 Face Swap Live — Google Colab (GPU)

**ANTES DE QUALQUER COISA:**
1. `Runtime → Change runtime type → T4 GPU → Save`
2. Execute as células **em ordem** (Shift+Enter)

> ⚠️ A **Célula 1** instala o onnxruntime-gpu e **reinicia o runtime automaticamente**.
> Após o restart, comece da **Célula 2** — não repita a Célula 1.

In [6]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 1 — Instala onnxruntime-gpu + REINICIA RUNTIME      ║
# ║  Execute UMA VEZ. Após o restart, vá para a Célula 2.       ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys

print('Removendo onnxruntime pré-instalado do Colab...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'onnxruntime', 'onnxruntime-gpu', 'onnxruntime-azure'],
               capture_output=True)

print('Instalando onnxruntime-gpu com suporte a CUDA...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'onnxruntime-gpu==1.20.1'],
    capture_output=True, text=True
)
print(r.stdout[-500:] if r.stdout else '')
if r.returncode != 0:
    print('ERRO:', r.stderr[-300:])
else:
    print('onnxruntime-gpu instalado!')

print()
print('Reiniciando runtime para limpar cache de imports...')
print('Após o restart, execute a partir da CÉLULA 2.')
get_ipython().kernel.do_shutdown(True)

Removendo onnxruntime pré-instalado do Colab...
Instalando onnxruntime-gpu com suporte a CUDA...

ERRO: 

Reiniciando runtime para limpar cache de imports...
Após o restart, execute a partir da CÉLULA 2.


{'status': 'ok', 'restart': True}

In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 2 — Verifica GPU + CUDA (após o restart)            ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess

r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode != 0:
    print('ERRO: GPU nao encontrada!')
    print('Va em Runtime → Change runtime type → T4 GPU e reinicie tudo.')
else:
    print(r.stdout)

import onnxruntime as ort
providers = ort.get_available_providers()
print('Providers ONNX disponíveis:', providers)

if 'CUDAExecutionProvider' in providers:
    print()
    print('✅ CUDA disponível! Pronto para GPU.')
else:
    print()
    print('❌ CUDA ainda não disponível.')
    print('   Tente: Runtime → Disconnect and delete runtime → mudar para T4 → rodar tudo de novo.')

Thu Jun 25 00:32:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 3 — Instala demais dependências + cloudflared       ║
# ╚══════════════════════════════════════════════════════════════╝
import os, subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'flask', 'flask-socketio',
     'insightface', 'opencv-python-headless',
     'huggingface_hub', 'pillow', 'numpy'],
    check=True
)
print('Dependências instaladas!')

# cloudflared: tunnel público sem precisar de conta
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared'], check=True)
    os.chmod('/usr/local/bin/cloudflared', 0o755)
    print('cloudflared instalado.')
else:
    print('cloudflared já existe.')

Dependências instaladas!
cloudflared já existe.


In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 4 — Clona / atualiza o repositório                  ║
# ╚══════════════════════════════════════════════════════════════╝
import os, subprocess

REPO = '/content/faceSwapLive2'

if not os.path.exists(REPO):
    subprocess.run(['git', 'clone',
        'https://github.com/moniquepavan/faceSwapLive2.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

os.chdir(REPO)
print('Diretório atual:', os.getcwd())
print('Arquivos:', os.listdir('.'))

Diretório atual: /content/faceSwapLive2
Arquivos: ['face_swap.py', 'app_colab.py', 'face_swap_gpu.py', 'baixar_modelo.py', 'models', 'requirements.txt', 'FaceSwapLive_Colab.ipynb', '.git', '__pycache__', 'instalar.bat', '.gitignore', 'README.md', 'templates', 'rodar.bat', 'app.py', 'faceSwapLive2']


In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 5 — Baixa o modelo inswapper (~280 MB)              ║
# ╚══════════════════════════════════════════════════════════════╝
import os, shutil
from huggingface_hub import hf_hub_download

DEST = '/content/faceSwapLive2/models/inswapper_128.onnx'
os.makedirs('/content/faceSwapLive2/models', exist_ok=True)

if os.path.exists(DEST):
    print(f'Modelo já existe ({os.path.getsize(DEST)/1e6:.0f} MB) — pulando.')
else:
    token = input('Cole seu token do Hugging Face (huggingface.co/settings/tokens): ').strip()
    print('Baixando...')
    path = hf_hub_download(
        repo_id='hacksider/deep-live-cam',
        filename='inswapper_128_fp16.onnx',
        token=token,
        local_dir='/content/faceSwapLive2/models'
    )
    if os.path.abspath(path) != os.path.abspath(DEST):
        shutil.move(path, DEST)
    print(f'Download concluído! {os.path.getsize(DEST)/1e6:.0f} MB')

Modelo já existe (278 MB) — pulando.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 6 — Inicia servidor Flask + tunnel cloudflared      ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, threading, time, sys, re, os

os.chdir('/content/faceSwapLive2')

# Para processos anteriores se houver
subprocess.run(['pkill', '-f', 'app_colab.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'],  capture_output=True)
time.sleep(1)

# Inicia Flask (threading mode, sem eventlet)
server = subprocess.Popen(
    [sys.executable, 'app_colab.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd='/content/faceSwapLive2'
)

def print_server_logs():
    for line in server.stdout:
        print('[srv]', line.rstrip())
threading.Thread(target=print_server_logs, daemon=True).start()

print('Aguardando Flask iniciar (8s)...')
time.sleep(8)

# Inicia tunnel cloudflared (sem conta, sem token)
cf = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print('Aguardando URL do tunnel (pode levar ~15s)...')
url = None
for line in cf.stdout:
    line = line.rstrip()
    print('[cf]', line)
    match = re.search(r'https://[\w\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

print()
print('=' * 60)
print(f'  ABRA NO NAVEGADOR: {url}')
print('=' * 60)
print()
print('Aguarde "Pronto! [CUDA GPU]" aparecer nos logs acima (~60s).')
print('Mantenha esta célula rodando. Ctrl+C para encerrar.')

try:
    cf.wait()
except KeyboardInterrupt:
    cf.terminate()
    server.terminate()
    print('Encerrado.')

Aguardando Flask iniciar (8s)...
[srv] Werkzeug appears to be used in a production deployment. Consider switching to a production web server instead.
[srv] /usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
[srv]   warnings.warn(
[srv] Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
[srv]  * Serving Flask app 'app_colab'
[srv]  * Debug mode: off
[srv] WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
[srv]  * Running on all addresses (0.0.0.0)
[srv]  * Running on http://127.0.0.1:5000
[srv]  * Running on http://172.28.0.12:5000
[srv] Press CTRL+C to quit
Aguardando URL do tunnel (pode levar ~15s)...
[cf] 2026-06-25T00:33:21Z INF Thank you for trying Cloudflare Tunnel. Doing so